#  Train/Validation/Test Split + Data Augmentation

**Autor:** Gema
**Fecha:** 2026
**Objetivo:** Dividir el dataset en 3 conjuntos y aumentar datos de entrenamiento

---

##  ORDEN IMPORTANTE
Dataset limpio (1,000 comentarios)
↓
1️⃣ SPLIT EN 3 CONJUNTOS
↙           ↓           ↘
Train (70%)  Val (15%)   Test (15%)
700          150          150
↓            ↓            ↓
2️⃣ AUGMENTATION  ❌ NO TOCAR  ❌ NO TOCAR
↓            ↓            ↓
Train (~1,000)  Val (150)   Test (150)
↓            ↓            ↓
Entrenar    Optuna       Evaluar

---

## 📚 ¿Qué es cada conjunto?

### 🟢 Train (70% = 700 comentarios)
- El modelo **APRENDE** con estos datos
- Es como los **apuntes** del alumno
- SÍ aplicamos augmentation
- Lo usamos para entrenar el modelo

### 🟡 Validation (15% = 150 comentarios)
- Se usa durante el entrenamiento
- Sirve para **ajustar hiperparámetros con Optuna**
- Es como los **exámenes de práctica**
- NO se toca con augmentation
- NUNCA se usa para evaluación final

### 🔴 Test (15% = 150 comentarios)
- Se usa SOLO al **final**
- Es como el **examen final**
- NO se toca con augmentation
- NUNCA se toca durante entrenamiento ni Optuna

---

## ⚠️ ¿Por qué el Split va PRIMERO?

Si hacemos augmentation ANTES del split ocurre **DATA LEAKAGE:**
Original (train):  "I hate you stupid idiot"
Aumentado (test):  "I hate you foolish idiot" ← casi igual ❌

El modelo "ya conoce" el test → métricas falsas → modelo malo en producción.

---

## 📊 Balance en los 3 Conjuntos

Usamos `stratify` para mantener el balance en los 3 conjuntos:
Train (700):

Normal (False): 376 (53.8%)
Tóxico (True):  324 (46.2%)

Validation (150):

Normal (False): 81 (53.8%)
Tóxico (True):  69 (46.2%)

Test (150):

Normal (False): 81 (53.8%)
Tóxico (True):  69 (46.2%)


---

## 🔄 Data Augmentation (SOLO en Train)

**Objetivo:** Aumentar train de 700 → ~1,000 comentarios
Train ANTES augmentation:

Normal: 376
Tóxico: 324
Total:  700

Train DESPUÉS augmentation:

Normal: 500 (+124 nuevos)
Tóxico: 500 (+176 nuevos)
Total: 1,000
Ratio: 1.0x ✅ PERFECTAMENTE BALANCEADO


---

##  Técnicas de Augmentation

### 🔴 Para TÓXICOS: SOLO Back-Translation

**¿Por qué SOLO back-translation?**

Synonym Replacement es PELIGROSO en textos tóxicos:
Original (TÓXICO):   "You are a stupid idiot"
Synonym Replacement: "You are a clever genius" ← Ya NO es tóxico ❌

Back-Translation mantiene el contexto completo:
Original (TÓXICO):  "You are a stupid idiot"
→ Español:          "Eres un idiota estúpido"
→ Inglés:           "You are a foolish idiot" ← SIGUE siendo tóxico ✅

### 🟢 Para NORMALES: Back-Translation + Synonym Replacement

En textos normales ambas técnicas son SEGURAS:
Original (NORMAL):   "This video is really interesting"
Synonym Replacement: "This video is really fascinating" ← SIGUE normal ✅

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from deep_translator import GoogleTranslator

# Configuración
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [2]:
# Cargar dataset limpio
print("="*80)
print("📂 CARGANDO DATASET LIMPIO")
print("="*80)

# Cargar archivo
df = pd.read_csv('../data/processed/youtoxic_comments_cleaned.csv')

print(f"✅ Dataset cargado correctamente")
print(f"\n📊 Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\n📋 Columnas: {df.columns.tolist()}")

# Verificar balance
print(f"\n BALANCE DEL TARGET:")
conteos = df['IsToxic'].value_counts()
porcentajes = df['IsToxic'].value_counts(normalize=True) * 100
print(f"  Normal (False): {conteos[False]:,} ({porcentajes[False]:.1f}%)")
print(f"  Tóxico (True):  {conteos[True]:,} ({porcentajes[True]:.1f}%)")
print(f"  Ratio: {max(conteos)/min(conteos):.2f}x")

📂 CARGANDO DATASET LIMPIO
✅ Dataset cargado correctamente

📊 Dimensiones: 1,000 filas × 5 columnas

📋 Columnas: ['CommentId', 'VideoId', 'Text', 'IsToxic', 'text_procesado']

 BALANCE DEL TARGET:
  Normal (False): 538 (53.8%)
  Tóxico (True):  462 (46.2%)
  Ratio: 1.16x


In [3]:
# Dividir dataset en Train / Validation / Test
print("="*80)
print(" DIVIDIENDO DATASET EN TRAIN / VALIDATION / TEST")
print("="*80)

# Preparar X e y
# X = texto procesado (input del modelo)
# y = IsToxic (lo que queremos predecir)
X = df['text_procesado']
y = df['IsToxic']

# PASO 1: Separar Train (70%) del resto (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,      # 30% para val+test
    random_state=42,     # Reproducibilidad
    stratify=y           # Mantener balance
)

# PASO 2: Separar Validation (15%) y Test (15%) del resto
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,      # 50% de 30% = 15% del total
    random_state=42,     # Reproducibilidad
    stratify=y_temp      # Mantener balance
)

# Mostrar resultados
print(f"\n RESULTADO DEL SPLIT:")
print(f"\n TRAIN:")
print(f"   Total: {len(X_train):,} comentarios ({len(X_train)/len(df)*100:.0f}%)")
print(f"   Normal: {sum(y_train==False):,} ({sum(y_train==False)/len(y_train)*100:.1f}%)")
print(f"   Tóxico: {sum(y_train==True):,} ({sum(y_train==True)/len(y_train)*100:.1f}%)")

print(f"\n VALIDATION:")
print(f"   Total: {len(X_val):,} comentarios ({len(X_val)/len(df)*100:.0f}%)")
print(f"   Normal: {sum(y_val==False):,} ({sum(y_val==False)/len(y_val)*100:.1f}%)")
print(f"   Tóxico: {sum(y_val==True):,} ({sum(y_val==True)/len(y_val)*100:.1f}%)")

print(f"\n TEST:")
print(f"   Total: {len(X_test):,} comentarios ({len(X_test)/len(df)*100:.0f}%)")
print(f"   Normal: {sum(y_test==False):,} ({sum(y_test==False)/len(y_test)*100:.1f}%)")
print(f"   Tóxico: {sum(y_test==True):,} ({sum(y_test==True)/len(y_test)*100:.1f}%)")

print(f"\n Split completado correctamente")
print(f" Balance mantenido en los 3 conjuntos")

 DIVIDIENDO DATASET EN TRAIN / VALIDATION / TEST

 RESULTADO DEL SPLIT:

 TRAIN:
   Total: 700 comentarios (70%)
   Normal: 377 (53.9%)
   Tóxico: 323 (46.1%)

 VALIDATION:
   Total: 150 comentarios (15%)
   Normal: 80 (53.3%)
   Tóxico: 70 (46.7%)

 TEST:
   Total: 150 comentarios (15%)
   Normal: 81 (54.0%)
   Tóxico: 69 (46.0%)

 Split completado correctamente
 Balance mantenido en los 3 conjuntos


In [5]:
# Guardar los 3 conjuntos en CSV
print("="*80)
print(" GUARDANDO LOS 3 CONJUNTOS")
print("="*80)

# Recuperar todas las columnas necesarias usando los índices
# X_train, X_val, X_test solo tienen 'text_procesado'
# Necesitamos recuperar el resto de columnas (CommentId, VideoId, Text, IsToxic)

# TRAIN
df_train = df.loc[X_train.index].copy()
df_train.to_csv('../data/processed/train.csv', index=False)
print(f"\n✅ Train guardado: data/processed/train.csv")
print(f"   Filas: {len(df_train):,}")
print(f"   Normal: {sum(df_train['IsToxic']==False):,} ({sum(df_train['IsToxic']==False)/len(df_train)*100:.1f}%)")
print(f"   Tóxico: {sum(df_train['IsToxic']==True):,} ({sum(df_train['IsToxic']==True)/len(df_train)*100:.1f}%)")

# VALIDATION
df_val = df.loc[X_val.index].copy()
df_val.to_csv('../data/processed/val.csv', index=False)
print(f"\n✅ Validation guardado: data/processed/val.csv")
print(f"   Filas: {len(df_val):,}")
print(f"   Normal: {sum(df_val['IsToxic']==False):,} ({sum(df_val['IsToxic']==False)/len(df_val)*100:.1f}%)")
print(f"   Tóxico: {sum(df_val['IsToxic']==True):,} ({sum(df_val['IsToxic']==True)/len(df_val)*100:.1f}%)")

# TEST
df_test = df.loc[X_test.index].copy()
df_test.to_csv('../data/processed/test.csv', index=False)
print(f"\n✅ Test guardado: data/processed/test.csv")
print(f"   Filas: {len(df_test):,}")
print(f"   Normal: {sum(df_test['IsToxic']==False):,} ({sum(df_test['IsToxic']==False)/len(df_test)*100:.1f}%)")
print(f"   Tóxico: {sum(df_test['IsToxic']==True):,} ({sum(df_test['IsToxic']==True)/len(df_test)*100:.1f}%)")

print(f"\n{'='*80}")
print(f" LOS 3 CONJUNTOS GUARDADOS CORRECTAMENTE")
print(f"{'='*80}")
print(f"\n Archivos en data/processed/:")
print(f"   - train.csv      ({len(df_train):,} filas) ← Augmentation después")
print(f"   - val.csv        ({len(df_val):,} filas)  ← NO tocar")
print(f"   - test.csv       ({len(df_test):,} filas)  ← NO tocar")

 GUARDANDO LOS 3 CONJUNTOS

✅ Train guardado: data/processed/train.csv
   Filas: 700
   Normal: 377 (53.9%)
   Tóxico: 323 (46.1%)

✅ Validation guardado: data/processed/val.csv
   Filas: 150
   Normal: 80 (53.3%)
   Tóxico: 70 (46.7%)

✅ Test guardado: data/processed/test.csv
   Filas: 150
   Normal: 81 (54.0%)
   Tóxico: 69 (46.0%)

 LOS 3 CONJUNTOS GUARDADOS CORRECTAMENTE

 Archivos en data/processed/:
   - train.csv      (700 filas) ← Augmentation después
   - val.csv        (150 filas)  ← NO tocar
   - test.csv       (150 filas)  ← NO tocar


In [6]:
from deep_translator import GoogleTranslator

# Traducir inglés → español
texto_español = GoogleTranslator(
    source='en',   # Idioma origen: inglés
    target='es'    # Idioma destino: español
).translate("I hate you stupid idiot")
# Resultado: "Te odio idiota estúpido"

# Traducir español → inglés
texto_ingles = GoogleTranslator(
    source='es',   # Idioma origen: español
    target='en'    # Idioma destino: inglés
).translate("Te odio idiota estúpido")
# Resultado: "I hate you foolish idiot"

In [7]:
import time

# Función de Back-Translation
def back_translation(texto, idioma_intermedio='es'):
    """
    Traduce un texto de inglés a otro idioma y de vuelta al inglés.
    
    Args:
        texto (str): Texto original en inglés
        idioma_intermedio (str): Idioma intermedio (por defecto español 'es')
    
    Returns:
        str: Texto traducido de vuelta al inglés
    """
    try:
        # PASO 1: Inglés → Español
        texto_intermedio = GoogleTranslator(
            source='en',
            target=idioma_intermedio
        ).translate(texto)
        
        # Pequeña pausa para no saturar la API
        time.sleep(0.5)
        
        # PASO 2: Español → Inglés
        texto_final = GoogleTranslator(
            source=idioma_intermedio,
            target='en'
        ).translate(texto_intermedio)
        
        return texto_final
    
    except Exception as e:
        # Si hay error, devolver el texto original
        print(f"⚠️ Error en back-translation: {e}")
        return texto

print("✅ Función back_translation() creada correctamente")
print("\n🧪 Probando con un ejemplo:")

# Probar con un ejemplo
ejemplo = "I hate you stupid idiot"
resultado = back_translation(ejemplo)
print(f"\nOriginal:  '{ejemplo}'")
print(f"Resultado: '{resultado}'")

✅ Función back_translation() creada correctamente

🧪 Probando con un ejemplo:

Original:  'I hate you stupid idiot'
Resultado: 'I hate you stupid idiot'


In [10]:
# Probar con comentarios REALES del dataset
print("="*80)
print("🧪 PROBANDO CON COMENTARIOS REALES DEL DATASET")
print("="*80)

# Cargar train
df_train = pd.read_csv('../data/processed/train.csv')

# Coger 3 comentarios tóxicos reales
toxicos = df_train[df_train['IsToxic'] == True]['text_procesado'].head(3)

for i, texto in enumerate(toxicos):
    print(f"\n--- Comentario {i+1} ---")
    print(f"Original:  '{texto[:100]}'")
    
    # Probar con alemán y árabe
    resultado_de = back_translation(texto[:200], idioma_intermedio='de')
    resultado_ar = back_translation(texto[:200], idioma_intermedio='ar')
    
    print(f"Alemán:    '{resultado_de[:100]}'")
    print(f"Árabe:     '{resultado_ar[:100]}'")
    time.sleep(1)

🧪 PROBANDO CON COMENTARIOS REALES DEL DATASET

--- Comentario 1 ---
Original:  'moron talking peaceful burning nation flag idiot even though thats suppose old knowledge baby making'
Alemán:    'Idiot talks peacefully and burns with national flag, idiot even if it is old knowledge, baby makes n'
Árabe:     'A peaceful talking moron who burns the country's flag and is an idiot even though this is supposed t'

--- Comentario 2 ---
Original:  'city full really really really dumb people cant try reason'
Alemán:    'The city is full, really, really, really stupid people can't try reason'
Árabe:     'The city is really full of people who are really stupid and can't experience the reason'

--- Comentario 3 ---
Original:  'okay blacklivesmovent thug stop carp u wanna stand murder yet child died gettin shoot cheast becasue'
Alemán:    'Okay, Blacklivesmovent, criminal, stop it, carp, you want to endure murder, but the child died, and '
Árabe:     'Well, black lives matter, thug movement, stop th

### CREAMOS COMENTARIOS TOXICOS CON BACK-TRANSLATION SOLO EN TRAIN

In [11]:
# Back-Translation para comentarios TÓXICOS
print("="*80)
print(" BACK-TRANSLATION - COMENTARIOS TÓXICOS")
print("="*80)

# Cargar train
df_train = pd.read_csv('../data/processed/train.csv')

# Separar tóxicos
df_toxicos = df_train[df_train['IsToxic'] == True].copy()

# Cuántos necesitamos generar
total_toxicos_objetivo = 500
toxicos_actuales = len(df_toxicos)
nuevos_necesarios = total_toxicos_objetivo - toxicos_actuales

print(f"\n Tóxicos actuales en train: {toxicos_actuales}")
print(f" Tóxicos objetivo: {total_toxicos_objetivo}")
print(f" Nuevos a generar: {nuevos_necesarios}")

# Seleccionar aleatoriamente cuáles traducir
df_toxicos_muestra = df_toxicos.sample(
    n=nuevos_necesarios,
    random_state=42
)

print(f"\n Aplicando back-translation a {nuevos_necesarios} comentarios...")
print(f" Esto puede tardar ~4 minutos. No cierres el notebook.\n")

# Aplicar back-translation
nuevos_textos = []
errores = 0

for i, (idx, fila) in enumerate(df_toxicos_muestra.iterrows()):
    try:
        # Traducir texto procesado
        texto_aumentado = back_translation(
            fila['text_procesado'][:300],  # Máximo 300 caracteres
            idioma_intermedio='de'          # Alemán
        )
        nuevos_textos.append(texto_aumentado)
        
        # Mostrar progreso cada 20 comentarios
        if (i + 1) % 20 == 0:
            print(f"   ✅ Procesados: {i+1}/{nuevos_necesarios}")
        
        time.sleep(0.5)  # Pausa para no saturar API
        
    except Exception as e:
        # Si hay error usar texto original
        nuevos_textos.append(fila['text_procesado'])
        errores += 1

print(f"\n✅ Back-translation completada")
print(f"⚠️ Errores encontrados: {errores}")

# Crear DataFrame con los nuevos comentarios tóxicos
df_toxicos_nuevos = df_toxicos_muestra.copy()
df_toxicos_nuevos['text_procesado'] = nuevos_textos
df_toxicos_nuevos['CommentId'] = [f'AUG_TOX_{i+1}' for i in range(nuevos_necesarios)]

print(f"\n Nuevos comentarios tóxicos generados: {len(df_toxicos_nuevos)}")
print(f"\n Ejemplo:")
print(f"Original:  '{df_toxicos_muestra.iloc[0]['text_procesado'][:80]}'")
print(f"Aumentado: '{df_toxicos_nuevos.iloc[0]['text_procesado'][:80]}'")

 BACK-TRANSLATION - COMENTARIOS TÓXICOS

 Tóxicos actuales en train: 323
 Tóxicos objetivo: 500
 Nuevos a generar: 177

 Aplicando back-translation a 177 comentarios...
 Esto puede tardar ~4 minutos. No cierres el notebook.

   ✅ Procesados: 20/177
   ✅ Procesados: 40/177
   ✅ Procesados: 60/177
   ✅ Procesados: 80/177
   ✅ Procesados: 100/177
   ✅ Procesados: 120/177
   ✅ Procesados: 140/177
   ✅ Procesados: 160/177

✅ Back-translation completada
⚠️ Errores encontrados: 0

 Nuevos comentarios tóxicos generados: 177

 Ejemplo:
Original:  'da fuck gay as bald white frenchy nigga gonna set hisself dang judge whole crimi'
Aumentado: 'da fuck gay like bald white frenchy nigga gonna fucking make himself the judge o'


### Para los comentarios NORMALES usamos DOS técnicas:

Back-Translation → 62 nuevos normales

Synonym Replacement → 62 nuevos normales

Total: 124 nuevos normales

In [12]:
# Back-Translation para comentarios NORMALES
print("="*80)
print(" BACK-TRANSLATION - COMENTARIOS NORMALES")
print("="*80)

# Separar normales
df_normales = df_train[df_train['IsToxic'] == False].copy()

# Cuántos necesitamos generar en total
total_normales_objetivo = 500
normales_actuales = len(df_normales)
nuevos_necesarios = total_normales_objetivo - normales_actuales

# Dividir entre las 2 técnicas
nuevos_back_translation = nuevos_necesarios // 2
nuevos_synonym = nuevos_necesarios - nuevos_back_translation

print(f"\n📊 Normales actuales en train: {normales_actuales}")
print(f"📊 Normales objetivo: {total_normales_objetivo}")
print(f"📊 Nuevos a generar: {nuevos_necesarios}")
print(f"   - Back-Translation: {nuevos_back_translation}")
print(f"   - Synonym Replacement: {nuevos_synonym}")

# Seleccionar aleatoriamente cuáles traducir con back-translation
df_normales_muestra_bt = df_normales.sample(
    n=nuevos_back_translation,
    random_state=42
)

print(f"\n⏳ Aplicando back-translation a {nuevos_back_translation} comentarios normales...")
print(f"⚠️ Esto puede tardar ~2 minutos. No cierres el notebook.\n")

# Aplicar back-translation
nuevos_textos_normales = []
errores = 0

for i, (idx, fila) in enumerate(df_normales_muestra_bt.iterrows()):
    try:
        texto_aumentado = back_translation(
            fila['text_procesado'][:300],
            idioma_intermedio='de'
        )
        nuevos_textos_normales.append(texto_aumentado)

        if (i + 1) % 20 == 0:
            print(f"   ✅ Procesados: {i+1}/{nuevos_back_translation}")

        time.sleep(0.5)

    except Exception as e:
        nuevos_textos_normales.append(fila['text_procesado'])
        errores += 1

print(f"\n✅ Back-translation completada")
print(f"⚠️ Errores encontrados: {errores}")

# Crear DataFrame con los nuevos comentarios normales (back-translation)
df_normales_bt = df_normales_muestra_bt.copy()
df_normales_bt['text_procesado'] = nuevos_textos_normales
df_normales_bt['CommentId'] = [f'AUG_NOR_BT_{i+1}' for i in range(nuevos_back_translation)]

print(f"\n📊 Nuevos comentarios normales (back-translation): {len(df_normales_bt)}")
print(f"\n🔎 Ejemplo:")
print(f"Original:  '{df_normales_muestra_bt.iloc[0]['text_procesado'][:80]}'")
print(f"Aumentado: '{df_normales_bt.iloc[0]['text_procesado'][:80]}'")

 BACK-TRANSLATION - COMENTARIOS NORMALES

📊 Normales actuales en train: 377
📊 Normales objetivo: 500
📊 Nuevos a generar: 123
   - Back-Translation: 61
   - Synonym Replacement: 62

⏳ Aplicando back-translation a 61 comentarios normales...
⚠️ Esto puede tardar ~2 minutos. No cierres el notebook.

   ✅ Procesados: 20/61
   ✅ Procesados: 40/61
   ✅ Procesados: 60/61

✅ Back-translation completada
⚠️ Errores encontrados: 0

📊 Nuevos comentarios normales (back-translation): 61

🔎 Ejemplo:
Original:  'forgot add timeline suspect also assaulted owner store threatens'
Aumentado: 'I forgot to add timeline. The suspect also attacked and threatened the shop owne'


In [ ]:
# OJO
# ⚠️ NOTA: Esta celda falló porque faltaban recursos de NLTK
# Se descargaron en la siguiente celda y se volvió a ejecutar
# Conservamos esta celda para documentar el proceso/ 

import nlpaug.augmenter.word as naw

# Crear augmentador de sinónimos
print("="*80)
print(" SYNONYM REPLACEMENT - COMENTARIOS NORMALES")
print("="*80)

# Crear augmentador
augmentador_sinonimos = naw.SynonymAug(
    aug_src='wordnet',
    aug_p=0.3
)

# Seleccionar comentarios normales que NO usamos en back-translation
df_normales_restantes = df_normales[
    ~df_normales.index.isin(df_normales_muestra_bt.index)
]

# Seleccionar cuántos necesitamos
df_normales_muestra_sr = df_normales_restantes.sample(
    n=nuevos_synonym,
    random_state=42
)

print(f"📊 Aplicando synonym replacement a {nuevos_synonym} comentarios...")

# Aplicar synonym replacement a TODOS los seleccionados
nuevos_textos_sr = []
errores_sr = 0

for i, (idx, fila) in enumerate(df_normales_muestra_sr.iterrows()):
    try:
        resultado = augmentador_sinonimos.augment(
            fila['text_procesado'][:300]
        )
        nuevos_textos_sr.append(resultado[0])

        if (i + 1) % 20 == 0:
            print(f"   ✅ Procesados: {i+1}/{nuevos_synonym}")

    except Exception as e:
        nuevos_textos_sr.append(fila['text_procesado'])
        errores_sr += 1

# Crear DataFrame con nuevos normales
df_normales_sr = df_normales_muestra_sr.copy()
df_normales_sr['text_procesado'] = nuevos_textos_sr
df_normales_sr['CommentId'] = [f'AUG_NOR_SR_{i+1}' for i in range(nuevos_synonym)]

print(f"\n✅ Synonym replacement completado")
print(f"⚠️ Errores: {errores_sr}")
print(f"📊 Nuevos normales generados: {len(df_normales_sr)}")
print(f"\n🔎 Ejemplo:")
print(f"Original:  '{df_normales_muestra_sr.iloc[0]['text_procesado'][:80]}'")
print(f"Resultado: '{df_normales_sr.iloc[0]['text_procesado'][:80]}'")



🟢 SYNONYM REPLACEMENT - COMENTARIOS NORMALES
📊 Aplicando synonym replacement a 62 comentarios...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is a


✅ Synonym replacement completado
⚠️ Errores: 62
📊 Nuevos normales generados: 62

🔎 Ejemplo:
Original:  'please slow cant tell sped m scully talking really quickly great video great inf'
Resultado: 'please slow cant tell sped m scully talking really quickly great video great inf'


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package aver

In [14]:
import nltk

# Descargar recursos necesarios para nlpaug
print("📥 Descargando recursos necesarios...")

nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("✅ Recursos descargados correctamente")

# Probar que funciona
augmentador_sinonimos = naw.SynonymAug(
    aug_src='wordnet',
    aug_p=0.3
)

ejemplo = "This video is really interesting and informative"
resultado = augmentador_sinonimos.augment(ejemplo)
print(f"\n🧪 Prueba:")
print(f"Original:  '{ejemplo}'")
print(f"Resultado: '{resultado[0]}'")

📥 Descargando recursos necesarios...


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gemit\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


✅ Recursos descargados correctamente

🧪 Prueba:
Original:  'This video is really interesting and informative'
Resultado: 'This television is really interesting and informative'


In [16]:
# Aplicar Synonym Replacement a los comentarios normales
print("="*80)
print("🟢 SYNONYM REPLACEMENT - COMENTARIOS NORMALES")
print("="*80)

# Seleccionar comentarios normales que NO usamos en back-translation
# ~ significa NOT (negación)
# isin() comprueba si el índice está en la lista
df_normales_restantes = df_normales[
    ~df_normales.index.isin(df_normales_muestra_bt.index)
]

# Seleccionar cuántos necesitamos
df_normales_muestra_sr = df_normales_restantes.sample(
    n=nuevos_synonym,
    random_state=42
)

print(f"\n📊 Comentarios seleccionados: {len(df_normales_muestra_sr)}")
print(f"⏳ Aplicando synonym replacement...")

# Aplicar synonym replacement
nuevos_textos_sr = []
errores_sr = 0

for i, (idx, fila) in enumerate(df_normales_muestra_sr.iterrows()):
    try:
        resultado = augmentador_sinonimos.augment(
            fila['text_procesado'][:300]
        )
        nuevos_textos_sr.append(resultado[0])

        if (i + 1) % 20 == 0:
            print(f"    Procesados: {i+1}/{nuevos_synonym}")

    except Exception as e:
        nuevos_textos_sr.append(fila['text_procesado'])
        errores_sr += 1

# Crear DataFrame con nuevos normales
df_normales_sr = df_normales_muestra_sr.copy()
df_normales_sr['text_procesado'] = nuevos_textos_sr
df_normales_sr['CommentId'] = [f'AUG_NOR_SR_{i+1}' for i in range(nuevos_synonym)]

print(f"\n Synonym replacement completado")
print(f" Errores: {errores_sr}")
print(f" Nuevos normales generados: {len(df_normales_sr)}")
print(f"\n Ejemplo:")
print(f"Original:  '{df_normales_muestra_sr.iloc[0]['text_procesado'][:80]}'")
print(f"Resultado: '{df_normales_sr.iloc[0]['text_procesado'][:80]}'")

🟢 SYNONYM REPLACEMENT - COMENTARIOS NORMALES

📊 Comentarios seleccionados: 62
⏳ Aplicando synonym replacement...
    Procesados: 20/62
    Procesados: 40/62
    Procesados: 60/62

 Synonym replacement completado
 Errores: 0
 Nuevos normales generados: 62

 Ejemplo:
Original:  'please slow cant tell sped m scully talking really quickly great video great inf'
Resultado: 'please slow cant tell sped m scully speak genuinely quickly great video enceinte'


### JUNTAMOS TODO EL DATASET AUMENTADO

In [17]:
# Juntar todo el dataset aumentado
print("="*80)
print("🔄 JUNTANDO TODO EL DATASET AUMENTADO")
print("="*80)

# Juntar todos los DataFrames
df_train_aumentado = pd.concat([
    df_train,           # 700 originales
    df_toxicos_nuevos,  # 177 tóxicos nuevos (back-translation)
    df_normales_bt,     # 62 normales nuevos (back-translation)
    df_normales_sr      # 61 normales nuevos (synonym replacement)
], ignore_index=True)

# Verificar resultado
print(f"\n📊 RESULTADO FINAL:")
print(f"\n   Originales:              700")
print(f"   Tóxicos nuevos (BT):     177")
print(f"   Normales nuevos (BT):     62")
print(f"   Normales nuevos (SR):     61")
print(f"   ─────────────────────────────")
print(f"   TOTAL:                 {len(df_train_aumentado):,}")

# Verificar balance
print(f"\n⚖️ BALANCE FINAL:")
conteos = df_train_aumentado['IsToxic'].value_counts()
porcentajes = df_train_aumentado['IsToxic'].value_counts(normalize=True) * 100
print(f"   Normal (False): {conteos[False]:,} ({porcentajes[False]:.1f}%)")
print(f"   Tóxico (True):  {conteos[True]:,} ({porcentajes[True]:.1f}%)")
print(f"   Ratio:          {max(conteos)/min(conteos):.2f}x")

🔄 JUNTANDO TODO EL DATASET AUMENTADO

📊 RESULTADO FINAL:

   Originales:              700
   Tóxicos nuevos (BT):     177
   Normales nuevos (BT):     62
   Normales nuevos (SR):     61
   ─────────────────────────────
   TOTAL:                 1,000

⚖️ BALANCE FINAL:
   Normal (False): 500 (50.0%)
   Tóxico (True):  500 (50.0%)
   Ratio:          1.00x


In [18]:
import os

# Guardar dataset aumentado
print("="*80)
print(" GUARDANDO DATASET AUMENTADO")
print("="*80)

# Ruta de salida
ruta_salida = '../data/processed/train_augmented.csv'

# Guardar
df_train_aumentado.to_csv(ruta_salida, index=False)

# Verificar que se guardó correctamente
if os.path.exists(ruta_salida):
    tamaño = os.path.getsize(ruta_salida) / 1024
    print(f"\n Dataset aumentado guardado correctamente")
    print(f"   Ruta: {ruta_salida}")
    print(f"   Tamaño: {tamaño:.1f} KB")
    print(f"   Filas: {len(df_train_aumentado):,}")
    print(f"   Columnas: {df_train_aumentado.columns.tolist()}")
else:
    print(f"\n ERROR: El archivo no se guardó correctamente")

# Resumen final de todos los archivos
print(f"\n{'='*80}")
print(f" ARCHIVOS FINALES EN data/processed/")
print(f"{'='*80}")

archivos = [
    ('train_augmented.csv', '1,000 filas', 'Entrenar el modelo ✅'),
    ('val.csv',             '150 filas',   'Ajustar Optuna ✅'),
    ('test.csv',            '150 filas',   'Evaluación final ✅'),
    ('train.csv',           '700 filas',   'Original sin aumentar (referencia)'),
    ('youtube_comments_cleaned.csv', '1,000 filas', 'Dataset limpio completo'),
]

for archivo, filas, uso in archivos:
    print(f"\n    {archivo}")
    print(f"      Filas: {filas}")
    print(f"      Uso:   {uso}")

 GUARDANDO DATASET AUMENTADO

 Dataset aumentado guardado correctamente
   Ruta: ../data/processed/train_augmented.csv
   Tamaño: 342.1 KB
   Filas: 1,000
   Columnas: ['CommentId', 'VideoId', 'Text', 'IsToxic', 'text_procesado']

 ARCHIVOS FINALES EN data/processed/

    train_augmented.csv
      Filas: 1,000 filas
      Uso:   Entrenar el modelo ✅

    val.csv
      Filas: 150 filas
      Uso:   Ajustar Optuna ✅

    test.csv
      Filas: 150 filas
      Uso:   Evaluación final ✅

    train.csv
      Filas: 700 filas
      Uso:   Original sin aumentar (referencia)

    youtube_comments_cleaned.csv
      Filas: 1,000 filas
      Uso:   Dataset limpio completo


##  RESUMEN DEL NOTEBOOK - SPLIT Y DATA AUGMENTATION

###  Resultados del Train/Validation/Test Split

**División del dataset original (1,000 comentarios):**

| Conjunto | Filas | Normal | Tóxico | Uso |
|----------|-------|--------|--------|-----|
| **Train** | 700 (70%) | 377 (53.8%) | 323 (46.2%) | Entrenar modelo |
| **Validation** | 150 (15%) | 81 (53.8%) | 69 (46.2%) | Ajustar Optuna |
| **Test** | 150 (15%) | 81 (53.8%) | 69 (46.2%) | Evaluación final |

**Técnica usada:** `train_test_split` con `stratify=y` para mantener balance en los 3 conjuntos.

---

###  Resultados de Data Augmentation (SOLO en Train)

**¿Por qué solo en Train?**
- Evitar **Data Leakage** (fuga de datos)
- Val y Test deben ser datos "nuevos" para el modelo
- Si aumentamos antes del split, el modelo vería datos similares en train y test

**Técnicas aplicadas:**

####  Para comentarios TÓXICOS: Solo Back-Translation
- **Idioma intermedio:** Alemán 🇩🇪
- **Razón:** Synonym Replacement es peligroso en tóxicos
  (puede cambiar palabras clave y el texto ya no sería tóxico)

####  Para comentarios NORMALES: Back-Translation + Synonym Replacement
- **Back-Translation:** Alemán 🇩🇪
- **Synonym Replacement:** WordNet (nlpaug, aug_p=0.3)
- **Razón:** Ambas técnicas son seguras en textos normales

**Resultado de la Augmentation:**

| Tipo | Originales | Nuevos | Total | Técnica |
|------|-----------|--------|-------|---------|
| **Tóxico** | 323 | 177 (BT) | 500 | Back-Translation |
| **Normal** | 377 | 62 (BT) + 61 (SR) | 500 | BT + Synonym |
| **TOTAL** | 700 | 300 | 1,000 | - |

---

###  Balance Final del Dataset Aumentado
Train Aumentado (train_augmented.csv):
Normal (False): 500 (50.0%) ✅
Tóxico (True):  500 (50.0%) ✅
Total:        1,000         ✅
Ratio:         1.00x        ✅ PERFECTAMENTE BALANCEADO

---

###  Archivos Generados

| Archivo | Filas | Uso |
|---------|-------|-----|
| `train_augmented.csv` | 1,000 | ✅ Entrenar el modelo |
| `val.csv` | 150 | ✅ Ajustar hiperparámetros con Optuna |
| `test.csv` | 150 | ✅ Evaluación final del modelo |
| `train.csv` | 700 | Referencia (original sin aumentar) |